In [3]:
!pip install pymorphy3

   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/8.4 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.4 MB ? eta -:--:--
   -- ------------------------------------- 0.5/8.4 MB 699.0 kB/s eta 0:00:12
   -- ------------------------------------- 0.5/8.4 MB 699.0 kB/s eta 0:00:12
   --- ------------------------------------ 0.8/8.4 MB 713.3 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/8.4 MB 713.3 kB/s eta 0:00:11
   --- ------------------------------------ 0.8/8.4 MB 713.3 kB/s eta 0:00:11
   ---- ----------------------------------- 1.0/8.4 MB 645.1 kB/s eta 0:00:12
   ---- ----------------------------------- 1.0/8.4 MB 645.1 kB/s eta 0:00:12
   ------ --------------------------------- 1.3/8.4 MB 645.3 kB/s eta 0:00:12
   ------ --------------------------------- 1.3/8.4 MB 645.3 kB/s eta 0:00:12
   ------ ------------

In [1]:
import pandas as pd
import numpy as np
import re
from collections import Counter
from pymorphy3 import MorphAnalyzer

morph = MorphAnalyzer()

In [2]:
df = pd.read_csv('spb_heritage.csv', sep=',')

In [3]:
df.head()

,number,name,is_ans,name_object,date_norm,date_buckets,date_buckets_2,author_norm,address,district_norm,protection_category_norm,base,id_object
0,1,NaN,0,Здание Консисторского управления Могилевской Р...,1870.0,1850-1874,1850-1899,В.И. Собольщиков;Е.С. Воротилов;Е.С. Воротилов...,"1-я Красноармейская ул., 11, лит. А, Б",1,1,Распоряжение КГИОП № 10-22 от 21.07.2009,669
1,2,NaN,0,Здание манежа (экзерциргауса) лейб-гвардии Изм...,1795.0,1775-1799,1750-1799,Кварнеги Дж,"1-я Красноармейская ул., 13; Измайловский пр.,...",1,1,Закон Санкт-Петербурга № 141-47 от 02.07.1997,1066
2,3,NaN,0,"Дом, где в начале 1895 г. Ленин В.И. встречалс...",NaN,NaN,NaN,NaN,"1-я Красноармейская ул., 22",1,1,Закон Санкт-Петербурга № 141-47 от 02.07.1997,1
3,4,NaN,0,Трансформаторная подстанция,1906.0,1900-1924,1900-1949,Горенберг Л.Б,"11-я Красноармейская ул., 28",1,1,Решение исполкома Ленгорсовета № 963 от 05.12....,2
4,5,NaN,0,Дом А.Н. Васильева,1912.0,1900-1924,1900-1949,Г.И. Котенков,"12-я Красноармейская ул., 3, лит. А",1,1,Распоряжение КГИОП № 229-рп от 28.09.2021; Рас...,697


In [4]:
df_name_id = df[['name_object', 'number']]
df_name_id.head()

,name_object,number
0,Здание Консисторского управления Могилевской Р...,1
1,Здание манежа (экзерциргауса) лейб-гвардии Изм...,2
2,"Дом, где в начале 1895 г. Ленин В.И. встречалс...",3
3,Трансформаторная подстанция,4
4,Дом А.Н. Васильева,5


In [5]:
words = []
for name in df_name_id['name_object']:
    name = re.sub(r'[^а-яА-Яa-zA-Z ]', '', name)
    words.extend(morph.parse(w)[0].normal_form for w in name.split(' ') if len(w) > 2)

words_cnt = Counter(words)
words_cnt.most_common(50)

[('дом', 2399),
 ('могила', 862),
 ('здание', 666),
 ('жилой', 598),
 ('корпус', 565),
 ('ограда', 381),
 ('флигель', 333),
 ('сад', 318),
 ('мост', 214),
 ('статуя', 213),
 ('особняк', 210),
 ('два', 207),
 ('советский', 201),
 ('церковь', 193),
 ('дача', 191),
 ('пруд', 190),
 ('завод', 186),
 ('жить', 182),
 ('казарма', 165),
 ('комплекс', 159),
 ('для', 156),
 ('герой', 155),
 ('парк', 141),
 ('год', 139),
 ('главное', 136),
 ('союз', 136),
 ('общество', 132),
 ('где', 130),
 ('служебный', 130),
 ('братский', 127),
 ('дворец', 124),
 ('ворота', 124),
 ('бюст', 122),
 ('производственный', 114),
 ('павильон', 111),
 ('доходный', 110),
 ('фонтан', 107),
 ('погибнуть', 107),
 ('великий', 106),
 ('главный', 97),
 ('война', 97),
 ('постройка', 93),
 ('канал', 80),
 ('памятник', 80),
 ('дот', 78),
 ('школа', 77),
 ('писатель', 74),
 ('двор', 74),
 ('отечественный', 70),
 ('который', 69)]

In [6]:
func_dict = {
    'могила' : 0,
    'жилой' : 0,
    'сад' : 0,
    'ограда' : 0,
    'мост' : 0,
    'статуя' : 0,
    'особняк' : 0,
    'церковь' : 0,
    'дача' : 0,
    'завод' : 0,
    'парк' : 0,
    'ворота' : 0,
    'бюст' : 0,
    'доходный' : 0,
    'фонтан' : 0
}

In [11]:
markers = []
for name, number in zip(df_name_id['name_object'], df_name_id['number']):
    name = re.sub(r'[^а-яА-Яa-zA-Z ]', '', name)
    
    line_mark = []
    for w in name.split(' '):

        if len(w) > 2:
            w = morph.parse(w)[0].normal_form

            if w in func_dict.keys():
                line_mark.append(w)
                func_dict[w] += 1

    markers.append(line_mark)

f_name_markers = df_name_id
f_name_markers['func_markers'] = markers

for k, v in func_dict.items():
    print(f'{k} -- {v}')

func_upd = []
for func, number in zip(f_name_markers['func_markers'], f_name_markers['number']):
    if func == []:
        func_upd.append(None)
    else:
        func_upd.append(','.join(func))

f_name_markers['func_markers'] = func_upd

могила -- 1724
жилой -- 1196
сад -- 636
ограда -- 762
мост -- 428
статуя -- 426
особняк -- 420
церковь -- 386
дача -- 382
завод -- 372
парк -- 282
ворота -- 248
бюст -- 244
доходный -- 220
фонтан -- 214


In [10]:
f_name_markers.sample(10)

,name_object,number,func_markers
4831,Производственный корпус,4832,NaN
8134,"""Мамкин"" сад",8135,сад
5434,Мост-плотина Большого пруда,5435,NaN
581,Сад,582,сад
6225,Бюст эллинистического поэта,6226,бюст
2405,Корпус профессорский со служебным флигелем,2406,NaN
6776,Мост Березовый через р.Кузьминку,6777,мост
5351,"Статуя ""Минерва""",5352,статуя
1051,Доходный дом,1052,доходный
6267,Здание приюта,6268,NaN


In [9]:
f_name_markers.to_csv('func_markers.csv', index=False, encoding='utf-8-sig')